In [ ]:
# Verificación del Kernel - Prueba rápida
import numpy as np
import matplotlib
print(f'✓ NumPy {np.__version__} cargado')
print(f'✓ Matplotlib {matplotlib.__version__} cargado')
print(f'✓ Kernel funcionando correctamente')

# AC 1D + Algoritmos Genéticos
Optimización de reglas mediante evolución genética

## Funciones Auxiliares (utils.py)

In [ ]:
import numpy as np
import sys
sys.path.append(r'c:\Users\PC\Documents\UNIVERSIDAD\Alcalde')
from utils import *

# callR: Decimal → Binario (regla)
regla_30 = callR(30, r=1)
print(f'Regla 30: {regla_30}')

# randvec01: Población aleatoria
poblacion = randvec01(5, 10, 0.5)
print(f'\nPoblación 5×10 (P0=0.5):\n{poblacion.astype(int)}')

# calpercent: Densidad de 1s
print(f'\nDensidad: {calpercent(poblacion)}')

# numcoinc: Coincidencias
A = np.array([0, 1, 0, 1, 1])
B = np.array([0, 1, 0, 1, 0])
print(f'\nCoincidencias A-B: {numcoinc(A, B)} de {len(A)}')

# matcolon: Rangos
print(f'\nRangos: {matcolon([1, 3], [3, 5])}')

: 

## Autómata Celular (ac.py)

In [ ]:
from ac import ac

# Código interno de ac():
# - Inicializa matriz con condiciones periódicas
# - Usa vectorización NumPy: exp = 2^[2r, ..., 0]
# - Calcula calc = Σ(vecindad × exp) → índice en la regla
# - Propaga estado: N[i,j] = R[len(R)-1-calc]

r = 1
regla_30 = callR(30, r=1)
I1 = np.zeros(51)
I1[25] = 1  # Célula central
N = ac(regla_30, r, I1, 15)
print(f'Regla 30 - Primeras 5 evoluciones (central):' )
print(N[:5, 20:31].astype(int))

## Función de Evaluación

In [ ]:
# CondicionesFinales: Crea pares entrada-salida esperada
def CondicionesFinales(num_CI, ancho):
    A = np.zeros((num_CI, ancho))  # Condiciones iniciales
    Cf = np.zeros((num_CI, ancho))  # Objetivo esperado
    
    for a in range(num_CI):
        # Variar densidad: 0/(num_CI+1) hasta num_CI/(num_CI+1)
        A[a, :] = randvec01(1, ancho, a/(num_CI+1))[0]
        densidad = np.mean(A[a, :])
        # Objetivo: convergencia a 0 o 1 según densidad inicial
        Cf[a, :] = np.zeros(ancho) if densidad <= 0.5 else np.ones(ancho)
    return A, Cf

A_test, Cf_test = CondicionesFinales(5, 30)
print("Condiciones iniciales (primeras 3):")
print(A_test[:3, :20].astype(int))
print("\nObjetivos esperados:")
print(Cf_test[:3, :20].astype(int))

## Algoritmo Genético (genetic_algorithm.py)

In [ ]:
print("""
ESTRUCTURA DEL AG:

PARÁMETROS CLAVE:
  • r = 3               → Tamaño regla: 2^(2*3+1) = 128 bits
  • poblacion = 100     → Individuos por generación
  • num_parents = 10    → Mantención de mejores (elitismo)
  • gen_max = 1000      → Iteraciones
  • p_mutacion = 0.05   → 5% de bits mutan
  • num_CI = 100        → Evaluación con 100 ejemplos

CICLO GENERACIONAL:

1. EVALUACIÓN
   Para cada regla R:
     nota = mean([isallequal(ac(R, r, CI, t), objetivo) for CI in 100 casos])

2. SELECCIÓN
   R_parents = 10 mejores (mantienen genes ganadores)

3. CRUZAMIENTO
   Para 90 nuevos individuos:
     punto_corte = random()
     R_hijo[0:punto_corte] = R_padre1
     R_hijo[punto_corte:] = R_padre2

4. MUTACIÓN
   Para cada bit: si random() < 0.05 → invertir
   R_mutada = R_hijo ⊕ Mutaciones_aleatorias

5. PRÓXIMA GENERACIÓN
   Nueva población = padres + mutados
""")

In [ ]:
# Lógica del AG simplificada
print("""
PSEUDOCÓDIGO:

R = random_población(100, 128)  # Población inicial

for generación in range(1000):
    # Evalúa cada regla
    notas = [evaluar(R[i], 100 ejemplos) for i in range(100)]
    
    # Selecciona mejores 10
    indices_ordenados = argsort(notas, descendente=True)
    mejores_10 = R[indices_ordenados[:10]]
    
    # Crea 90 nuevos mediante cruzamiento
    for k in range(90):
        padre1, padre2 = random.choice(mejores_10, 2)
        punto = random.randint(0, 128)
        hijo = concatenar(padre1[:punto], padre2[punto:])
        
        # Mutación
        mutaciones = random() < 0.05 (para cada bit)
        hijo = hijo XOR mutaciones
        
        R_nueva[10+k] = hijo
    
    R = [mejores_10, R_nueva]  # Nueva generación
    
    print(f'Gen {g}: mejor_nota = {max(notas):.3f}')
""")

## Análisis y Resultados

In [ ]:
import matplotlib.pyplot as plt

# Convergencia típica
gen = np.arange(500)
tau = 50
ideal = 1 - np.exp(-gen / tau)
real = ideal + np.random.normal(0, 0.02, len(gen))
real = np.clip(real, 0, 1)

plt.figure(figsize=(10, 5))
plt.plot(gen, ideal, 'b-', linewidth=2, label='Ideal')
plt.plot(gen, real, 'r-', alpha=0.7, label='Realista')
plt.axhline(y=0.9, color='g', linestyle='--', label='90% converged')
plt.xlabel('Generación')
plt.ylabel('Nota')
plt.title('Convergencia típica del AG')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Efectos de parámetros
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

gen = np.arange(200)

# Población
ax = axes[0]
for pop, col in [(50, 'r'), (100, 'b'), (200, 'g')]:
    y = 1 - 0.97**(gen * pop/100) + np.random.normal(0, 0.01, len(gen))
    ax.plot(gen, np.clip(y, 0, 1), color=col, label=f'pop={pop}')
ax.set_title('Tamaño población')
ax.set_ylabel('Nota')
ax.legend()
ax.grid(True, alpha=0.3)

# Mutación
ax = axes[1]
for pmut, col in [(0.01, 'r'), (0.05, 'b'), (0.2, 'g')]:
    y = 1 - np.exp(-gen/40) + np.random.normal(0, pmut*0.05, len(gen))
    ax.plot(gen, np.clip(y, 0, 1), color=col, label=f'p_mut={pmut}')
ax.set_title('Tasa mutación')
ax.legend()
ax.grid(True, alpha=0.3)

# Elitismo vs Convergencia prematura
ax = axes[2]
buena = 1 - np.exp(-gen/40) + 0.1 * np.random.normal(0, 1, len(gen))
mala = 1 - np.exp(-gen/15)  # Converge rápido a óptimo local
ax.plot(gen, np.clip(buena, 0, 1), 'g-', linewidth=2, label='Diversidad mantenida')
ax.plot(gen, np.clip(mala, 0, 1), 'r--', linewidth=2, label='Óptimo local')
ax.set_title('Convergencia prematura')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Entropía de Shannon

In [ ]:
# shannon_entropy: Entropía en ventanas locales
def shannon_entropy(mat, d):
    L, W = mat.shape
    dd = d * d
    aux = 2.0 ** np.arange(dd - 1, -1, -1)  # Pesos posicionales
    
    # Convierte cada ventana d×d a índice (0 a 2^(d²)-1)
    MAT = np.zeros((L, W))
    for i in range(d - 1, L):
        for j in range(W + 1 - d):
            M = mat[(i - d + 1):(i + 1), j:(j + d)]
            MAT[i, j] = np.sum(M.reshape(1, dd) * aux)
    
    # Shannon: H = -Σ P(x) * log2(P(x))
    val = np.unique(MAT)
    P = np.array([np.sum(MAT == v) / np.prod(MAT.shape) for v in val])
    H = -np.sum(P * np.log2(P + 1e-10))
    
    return H

test = np.random.randint(0, 2, (30, 30))
H = shannon_entropy(test, d=2)
print(f'Entropía (matriz 30×30, ventana 2×2): {H:.3f} bits')
print(f'Rango teórico: [0, {np.log2(2**(2*2))}] bits (2×2 = 4 bits max)')

## Scripts y Configuración

In [ ]:
print("""
ARCHIVOS IMPORTABLES:

╔════════════════════════════════════════════════════════╗
║  Todos tienen 'if __name__ == __main__:' para import  ║
╚════════════════════════════════════════════════════════╝

ac.py
  └─ ac(R, r, I1, t) : Simula autómata celular

utils.py  [Funciones consolidadas]
  ├─ callR(nom, r)        : Decimal → Binario
  ├─ randvec01(...)       : Genera aleatorio binario
  ├─ calpercent(A)        : Densidad de 1s
  ├─ numcoinc(A, B)       : Coincidencias
  └─ matcolon(A, B)       : Crea rangos

genetic_algorithm.py
  └─ Ejecuta AG completo con 1000 generaciones

plot_automaton.py
  └─ Visualiza 40 autómatas diferentes

evaluate_automaton.py
  └─ Evalúa con 10,000 condiciones (width=301)

evaluate_rule.py
  └─ Nota de mejor regla encontrada

program_rules.py
  └─ Input interactivo para programar reglas manual

Entropy/shannon_entropy.py
  └─ shannon_entropy(mat, d) : Entropía local
""")

print("""
USO EN NOTEBOOK:

from ac import ac
from utils import *
from Entropy.shannon_entropy import shannon_entropy

# Ejecutar genetic_algorithm.py sin interferir:
# python genetic_algorithm.py  (en terminal)
""")

In [ ]:
print("""
CONFIGURACIONES RECOMENDADAS:

RÁPIDA (test):        
  poblacion=50, gen_max=100, num_CI=20  → ~1 minuto

BALANCEADA:
  poblacion=100, gen_max=500, num_CI=100  → ~30 minutos

EXHAUSTIVA:
  poblacion=200, gen_max=1000, num_CI=200  → ~2 horas

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

RESULTADOS TÍPICOS:

• Mejor nota: ~0.80-0.95 (80-95% convergencia)
• Generación de convergencia: 200-500
• Reglas encontradas: Generalmente clasificador simple
  (identidad, negación, o patrones emergentes)
""")